In [4]:
import os
import cv2
import numpy as np
import pandas as pd
import zipfile
from io import BytesIO
from tqdm import tqdm  # For progress bars

In [5]:
# --- Configuration ---
INPUT_DIR = '/kaggle/input/dataset-patching/dataset/'
OUTPUT_DIR = '/kaggle/working/'
PATCH_SIZE = 224
STRIDE = 224  # Step size. If equal to PATCH_SIZE, no overlap.
TISSUE_THRESHOLD = 0.1  # Keep patch if at least 10% of it is tissue (according to mask)

def create_patches(image, mask, patch_size, stride, threshold):
    """
    Slices an image into patches and filters them based on the mask.
    Returns: List of image patches, List of mask patches
    """
    img_h, img_w = image.shape[:2]
    patches_img = []
    patches_mask = []
    
    # Iterate over the image with the sliding window
    for y in range(0, img_h - patch_size + 1, stride):
        for x in range(0, img_w - patch_size + 1, stride):
            
            # Crop the mask patch first to check if it's useful
            mask_patch = mask[y:y+patch_size, x:x+patch_size]
            
            # Calculate tissue percentage (assuming mask is 0 or 255/1)
            # If your mask is 0-255, divide by 255 first or change logic
            tissue_ratio = np.mean(mask_patch > 0) 
            
            if tissue_ratio >= threshold:
                # If useful, crop the image patch
                img_patch = image[y:y+patch_size, x:x+patch_size]
                
                patches_img.append(img_patch)
                patches_mask.append(mask_patch)
                
    return patches_img, patches_mask

In [6]:
# --- Load Labels ---
train_df = pd.read_csv(os.path.join(INPUT_DIR, 'train_labels.csv'))
# Map text labels to integers immediately
label_mapping = {
    'Luminal A': 0, 'Luminal B': 1, 'HER2(+)': 2, 'Triple negative': 3
}
train_df['target'] = train_df['label'].map(label_mapping)

# --- Processing Lists ---
X_train_patches = []
y_train_patches = []
train_groups = [] # To track which original image the patch came from (for splitting)
train_masks_patches = []

In [7]:
# --- 1. Process Training Data ---
print("Processing Training Data (from folders)...")

# Define folder paths
train_img_dir = os.path.join(INPUT_DIR, 'train_data')
train_mask_dir = os.path.join(DATA_DIR, 'train_masks')

for index, row in tqdm(train_df.iterrows(), total=len(train_df)):
    filename = row['sample_index'] # e.g., "img_0001.png"
    label = row['target']
    
    # Build full paths
    img_path = os.path.join(train_img_dir, filename)
    mask_path = os.path.join(train_mask_dir, filename)
    
    # Check if file exists
    if not os.path.exists(img_path):
        print(f"Warning: {filename} not found.")
        continue
        
    # Read Image & Mask
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert BGR to RGB
    
    # Read mask as grayscale
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    # Generate Patches
    p_imgs, p_masks = create_patches(image, mask, PATCH_SIZE, STRIDE, TISSUE_THRESHOLD)
    
    # Store
    for p_img, p_mask in zip(p_imgs, p_masks):
        X_train_patches.append(p_img)
        train_masks_patches.append(p_mask)
        y_train_patches.append(label)
        train_groups.append(filename)

# Convert to Numpy Arrays
X_train = np.array(X_train_patches)
y_train = np.array(y_train_patches)
train_masks = np.array(train_masks_patches)
train_groups = np.array(train_groups)

# Add channel dim to masks
if len(train_masks.shape) == 3:
    train_masks = np.expand_dims(train_masks, axis=-1)

print(f"Generated {len(X_train)} training patches.")

# Save Training Data
np.save(os.path.join(OUTPUT_DIR, 'train_patches_images.npy'), X_train)
np.save(os.path.join(OUTPUT_DIR, 'train_patches_masks.npy'), train_masks)
np.save(os.path.join(OUTPUT_DIR, 'train_patches_labels.npy'), y_train)
np.save(os.path.join(OUTPUT_DIR, 'train_patches_groups.npy'), train_groups)

print("\n✅ Patching Train Set completed. Files saved.")

Processing Training Data (from folders)...


NameError: name 'DATA_DIR' is not defined

In [ ]:
# --- 2. Process Test Data ---
print("\nProcessing Test Data (from folders)...")

X_test_patches = []
test_masks_patches = []
test_groups = []

test_img_dir = os.path.join(DATA_DIR, 'test_images')
test_mask_dir = os.path.join(DATA_DIR, 'test_masks')

# Get all png files in test folder
test_files = [f for f in os.listdir(test_img_dir) if f.endswith('.png')]

for filename in tqdm(test_files):
    img_path = os.path.join(test_img_dir, filename)
    mask_path = os.path.join(test_mask_dir, filename)
    
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Handle mask (create dummy if missing)
    if os.path.exists(mask_path):
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    else:
        # If mask is missing, assume essentially the whole image is valid
        mask = np.ones((image.shape[0], image.shape[1]), dtype=np.uint8) * 255

    # Generate Patches
    p_imgs, p_masks = create_patches(image, mask, PATCH_SIZE, STRIDE, TISSUE_THRESHOLD)
    
    for p_img, p_mask in zip(p_imgs, p_masks):
        X_test_patches.append(p_img)
        test_masks_patches.append(p_mask)
        test_groups.append(filename)

X_test = np.array(X_test_patches)
test_masks = np.array(test_masks_patches)
test_groups = np.array(test_groups)

if len(test_masks.shape) == 3:
    test_masks = np.expand_dims(test_masks, axis=-1)

print(f"Generated {len(X_test)} test patches.")

# Save Test Data
np.save(os.path.join(OUTPUT_DIR, 'test_patches_images.npy'), X_test)
np.save(os.path.join(OUTPUT_DIR, 'test_patches_masks.npy'), test_masks)
np.save(os.path.join(OUTPUT_DIR, 'test_patches_groups.npy'), test_groups)

print("\n✅ Patching Test Set complete. Files saved.")